In [4]:
# =========================
# IMPORTS
# =========================
import pandas as pd
import numpy as np

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

# =========================
# LOAD DATASET
# =========================
file_path = "merged_final_v3.csv"   # 🔁 change if needed
df = pd.read_csv(file_path)

print("✅ Dataset loaded")
print("Shape:", df.shape)

# =========================
# CHECK LABEL COLUMN
# =========================
if "label" not in df.columns:
    raise ValueError("❌ 'label' column not found in dataset")

# =========================
# SEPARATE LABEL
# =========================
y = df["label"]

# Encode labels (tumor=1, normal=0)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))

# =========================
# CLEAN FEATURES
# =========================
X = df.drop(columns=["label"])

# Remove non-numeric columns (VERY IMPORTANT)
X = X.select_dtypes(include=[np.number])

print("✅ Numeric features:", X.shape[1])

# =========================
# HANDLE MISSING VALUES
# =========================
missing_before = X.isnull().sum().sum()
X = X.fillna(X.mean())
missing_after = X.isnull().sum().sum()

print(f"Missing values before: {missing_before}")
print(f"Missing values after: {missing_after}")

# =========================
# NORMALIZATION
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("✅ Data normalized")

# =========================
# RFE SETUP
# =========================
n_features = min(500, X_scaled.shape[1])  # safe check

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rfe = RFE(
    estimator=model,
    n_features_to_select=n_features,
    step=50   # increase to 100 if slow
)

print("⏳ Running RFE... (this may take time)")
X_selected = rfe.fit_transform(X_scaled, y_encoded)

# =========================
# GET SELECTED GENES
# =========================
selected_genes = X.columns[rfe.support_]

print(f"✅ Selected {len(selected_genes)} genes")

# =========================
# FINAL DATASET
# =========================
final_df = pd.DataFrame(X_selected, columns=selected_genes)
final_df["label"] = y.values

print("Final dataset shape:", final_df.shape)

# =========================
# SAVE FILE
# =========================
output_file = "selected_500_genes_normalized.csv"
final_df.to_csv(output_file, index=False)

print(f"🎉 SUCCESS: File saved as '{output_file}'")

✅ Dataset loaded
Shape: (6322, 16482)
Classes: [np.int64(0), np.int64(1)]
✅ Numeric features: 16479
Missing values before: 0
Missing values after: 0
✅ Data normalized
⏳ Running RFE... (this may take time)
✅ Selected 500 genes
Final dataset shape: (6322, 501)
🎉 SUCCESS: File saved as 'selected_500_genes_normalized.csv'
